In [4]:
import random
import copy
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# ==========================================
# 1. ESTRUCTURA DEL ÁRBOL GENÉTICO
# ==========================================
class Nodo:
    def __init__(self, valor, hijos=None):
        self.valor = valor
        self.hijos = hijos if hijos else []

    def __str__(self):
        if not self.hijos: return self.valor
        return f"{self.valor}({', '.join([str(c) for c in self.hijos])})"

TERMINALES = ['Avanzar', 'Girar_Izquierda', 'Girar_Derecha', 'Entregar_Galleta', 'Mirar_Hacia_Ingeniero']
FUNCIONES = {'Secuencia': 2, 'Repetir_3': 1, 'Si_Tengo_Galletas': 2}

def generar_arbol(profundidad, max_profundidad=4):
    """Crea un programa aleatorio."""
    if profundidad >= max_profundidad or (profundidad > 1 and random.random() < 0.3):
        return Nodo(random.choice(TERMINALES))
    else:
        func = random.choice(list(FUNCIONES.keys()))
        return Nodo(func, [generar_arbol(profundidad + 1, max_profundidad) for _ in range(FUNCIONES[func])])

def obtener_nodos(nodo):
    nodos = [nodo]
    for hijo in nodo.hijos: nodos.extend(obtener_nodos(hijo))
    return nodos

def cruzar(padre1, padre2):
    """Intercambia ramas entre dos programas."""
    h1, h2 = copy.deepcopy(padre1), copy.deepcopy(padre2)
    n1, n2 = obtener_nodos(h1), obtener_nodos(h2)
    if len(n1) > 1 and len(n2) > 1:
        p1, p2 = random.choice(n1[1:]), random.choice(n2[1:])
        p1.valor, p2.valor = p2.valor, p1.valor
        p1.hijos, p2.hijos = p2.hijos, p1.hijos
    return h1, h2

def mutar(nodo, max_profundidad=4):
    """Sustituye una rama aleatoria por una nueva."""
    n = copy.deepcopy(nodo)
    nodos = obtener_nodos(n)
    if len(nodos) > 1:
        target = random.choice(nodos[1:])
        nuevo_subarbol = generar_arbol(0, max_profundidad)
        target.valor, target.hijos = nuevo_subarbol.valor, nuevo_subarbol.hijos
    else:
        return generar_arbol(0, max_profundidad)
    return n

# ==========================================
# 2. MOTOR DE SIMULACIÓN Y EVALUACIÓN
# ==========================================
def ejecutar(nodo, estado):
    if estado['pasos'] <= 0: return

    v = nodo.valor
    if v in TERMINALES:
        estado['pasos'] -= 1
        if v == 'Avanzar':
            d = estado['dir']
            if d == 0: estado['y'] = min(estado['y'] + 1, 9)       # Norte
            elif d == 1: estado['x'] = min(estado['x'] + 1, 9)     # Este
            elif d == 2: estado['y'] = max(estado['y'] - 1, 0)     # Sur
            elif d == 3: estado['x'] = max(estado['x'] - 1, 0)     # Oeste
        elif v == 'Girar_Izquierda': estado['dir'] = (estado['dir'] - 1) % 4
        elif v == 'Girar_Derecha': estado['dir'] = (estado['dir'] + 1) % 4
        elif v == 'Mirar_Hacia_Ingeniero':
            pendientes = [ing for ing in estado['ingenieros'] if not ing['recibido']]
            if pendientes:
                cercano = min(pendientes, key=lambda ing: abs(ing['x']-estado['x']) + abs(ing['y']-estado['y']))
                dx, dy = cercano['x'] - estado['x'], cercano['y'] - estado['y']
                if abs(dx) > abs(dy): estado['dir'] = 1 if dx > 0 else 3
                else:
                    if dy != 0: estado['dir'] = 0 if dy > 0 else 2
        elif v == 'Entregar_Galleta':
            if estado['galletas'] > 0:
                for ing in estado['ingenieros']:
                    if not ing['recibido'] and abs(ing['x']-estado['x']) + abs(ing['y']-estado['y']) <= 1:
                        ing['recibido'] = True
                        estado['galletas'] -= 1
                        estado['puntos'] += 100
                        break

        estado['visitados'].add((estado['x'], estado['y']))

        estado['history'].append({
            'x': estado['x'], 'y': estado['y'], 'dir': estado['dir'],
            'galletas': estado['galletas'], 'puntos': estado['puntos'],
            'ingenieros': copy.deepcopy(estado['ingenieros'])
        })
    else:
        if v == 'Secuencia':
            ejecutar(nodo.hijos[0], estado)
            if estado['pasos'] > 0: ejecutar(nodo.hijos[1], estado)
        elif v == 'Repetir_3':
            for _ in range(3):
                if estado['pasos'] > 0: ejecutar(nodo.hijos[0], estado)
        elif v == 'Si_Tengo_Galletas':
            if estado['galletas'] > 0: ejecutar(nodo.hijos[0], estado)
            else: ejecutar(nodo.hijos[1], estado)

def evaluar(programa, ingenieros_coords):
    estado = {
        'x': 5, 'y': 5, 'dir': 0, 'galletas': 10, 'puntos': 0, 'pasos': 80,
        'ingenieros': [{'x': x, 'y': y, 'recibido': False} for x,y in ingenieros_coords],
        'visitados': {(5,5)},
        'history': []
    }
    estado['history'].append({
        'x': 5, 'y': 5, 'dir': 0, 'galletas': 10, 'puntos': 0,
        'ingenieros': copy.deepcopy(estado['ingenieros'])
    })

    while estado['pasos'] > 0:
        prev_pasos = estado['pasos']
        ejecutar(programa, estado)
        if estado['pasos'] == prev_pasos:
            break

    entregadas = sum(1 for ing in estado['ingenieros'] if ing['recibido'])
    aptitud = (entregadas * 100) + len(estado['visitados'])
    return aptitud, estado

# ==========================================
# 3. ALGORITMO GENÉTICO (Evolución)
# ==========================================
print("Generando sala e ingenieros aleatorios...")
coords_ingenieros = [(random.randint(0,9), random.randint(0,9)) for _ in range(8)]
poblacion = [generar_arbol(0, 4) for _ in range(300)]

print("Iniciando evolución del cerebro del robot...\n")
for gen in range(20):
    aptitudes = [(p, evaluar(p, coords_ingenieros)[0]) for p in poblacion]
    aptitudes.sort(key=lambda x: x[1], reverse=True)

    print(f"Generación {gen:02d} | Mejor Aptitud: {aptitudes[0][1]} puntos")

    nueva_poblacion = [aptitudes[i][0] for i in range(5)]
    while len(nueva_poblacion) < 300:
        p1 = random.choice(aptitudes[:50])[0]
        p2 = random.choice(aptitudes[:50])[0]
        h1, h2 = cruzar(p1, p2)
        if random.random() < 0.2: h1 = mutar(h1)
        if random.random() < 0.2: h2 = mutar(h2)
        nueva_poblacion.extend([h1, h2])
    poblacion = nueva_poblacion[:300]

mejor_programa = aptitudes[0][0]
mejor_aptitud, mejor_estado = evaluar(mejor_programa, coords_ingenieros)

print("\n" + "="*50)
print(f"🧬 ADN DEL MEJOR ROBOT:\n{mejor_programa}")
print("="*50)

# ==========================================
# 4. VISUALIZACIÓN GRÁFICA ROBUSTA
# ==========================================
print("\nGenerando video interactivo... (Espera un momento)")

def generar_video(estado):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, 9.5)
    ax.set_ylim(-0.5, 9.5)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    ax.text(4.5, 10.2, 'Robot Repartidor', fontsize=16, ha='center', weight='bold')
    info = ax.text(4.5, 9.5, '', fontsize=12, ha='center', color='blue')

    # Elementos gráficos garantizados en cualquier entorno
    robot_marker, = ax.plot([], [], marker='^', color='blue', markersize=15, linestyle='None', zorder=10, label='Robot')
    hungry_marker, = ax.plot([], [], marker='s', color='red', markersize=12, linestyle='None', label='Esperando')
    fed_marker, = ax.plot([], [], marker='o', color='green', markersize=12, linestyle='None', label='Con Galleta')

    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1))

    historial = estado['history']

    def init():
        info.set_text('')
        robot_marker.set_data([], [])
        hungry_marker.set_data([], [])
        fed_marker.set_data([], [])
        return robot_marker, hungry_marker, fed_marker, info

    def update(frame):
        s = historial[frame]

        # Marcadores direccionales: 0:Norte(^), 1:Este(>), 2:Sur(v), 3:Oeste(<)
        direcciones = ['^', '>', 'v', '<']
        robot_marker.set_marker(direcciones[s['dir']])
        robot_marker.set_data([s['x']], [s['y']])

        # Separar ingenieros por estado
        hx, hy, fx, fy = [], [], [], []
        for ing in s['ingenieros']:
            if ing['recibido']:
                fx.append(ing['x'])
                fy.append(ing['y'])
            else:
                hx.append(ing['x'])
                hy.append(ing['y'])

        hungry_marker.set_data(hx, hy)
        fed_marker.set_data(fx, fy)

        info.set_text(f"Puntos: {s['puntos']} | Galletas: {s['galletas']} | Pasos: {frame}")
        return robot_marker, hungry_marker, fed_marker, info

    anim = FuncAnimation(fig, update, frames=len(historial), init_func=init, blit=True, interval=150)
    plt.close()
    return HTML(anim.to_jshtml())

# Mostrar el video
display(generar_video(mejor_estado))

Generando sala e ingenieros aleatorios...
Iniciando evolución del cerebro del robot...

Generación 00 | Mejor Aptitud: 630 puntos
Generación 01 | Mejor Aptitud: 827 puntos
Generación 02 | Mejor Aptitud: 827 puntos
Generación 03 | Mejor Aptitud: 833 puntos
Generación 04 | Mejor Aptitud: 835 puntos
Generación 05 | Mejor Aptitud: 835 puntos
Generación 06 | Mejor Aptitud: 836 puntos
Generación 07 | Mejor Aptitud: 836 puntos
Generación 08 | Mejor Aptitud: 836 puntos
Generación 09 | Mejor Aptitud: 836 puntos
Generación 10 | Mejor Aptitud: 836 puntos
Generación 11 | Mejor Aptitud: 844 puntos
Generación 12 | Mejor Aptitud: 844 puntos
Generación 13 | Mejor Aptitud: 844 puntos
Generación 14 | Mejor Aptitud: 844 puntos
Generación 15 | Mejor Aptitud: 844 puntos
Generación 16 | Mejor Aptitud: 844 puntos
Generación 17 | Mejor Aptitud: 844 puntos
Generación 18 | Mejor Aptitud: 844 puntos
Generación 19 | Mejor Aptitud: 844 puntos

🧬 ADN DEL MEJOR ROBOT:
Secuencia(Secuencia(Secuencia(Secuencia(Entregar